# RISK ANALYTICS — 02 Ingesta Bronze

**Fuente:** `/Volumes/eventosriesgoyvar/default/bdeventos/BD eventos riesgo operacional.csv`
**Destino:** `eventosriesgoyvar.bronze.eventos_riesgo_raw`

Reglas de esta capa:
- **Copia fiel:** las 58 columnas del CSV se cargan como string, sin limpieza; no se elimina ningún registro. Las filas con valores idénticos no se consideran duplicadas (estructura de la BD).
- **Única adaptación:** nombres de columna normalizados a snake_case (Delta no acepta espacios, tildes, paréntesis ni puntos). Los **valores** quedan tal cual el archivo.
- **Trazabilidad:** `_archivo_origen`, `_fecha_ingesta` y `_lote_procesamiento`.
- **Idempotencia:** si el archivo ya fue ingerido, la celda de ingesta no inserta de nuevo (evita duplicados por la misma carga).

In [0]:
from pyspark.sql import functions as F
import re
from datetime import datetime

# Rutas y nombres definidos en el diseño aprobado
RUTA_CSV = "/Volumes/eventosriesgoyvar/default/bdeventos/BD eventos riesgo operacional.csv"
NOMBRE_ARCHIVO = RUTA_CSV.split("/")[-1]
TABLA_BRONZE = "eventosriesgoyvar.bronze.eventos_riesgo_raw"

LOTE = f"bronze_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
FECHA_INGESTA = datetime.now()

spark.sql("CREATE SCHEMA IF NOT EXISTS eventosriesgoyvar.bronze COMMENT 'Capa Bronze: copia cruda de eventos de riesgo operacional'")

print("Archivo fuente:", NOMBRE_ARCHIVO)
print("Lote de procesamiento:", LOTE)
print("Tabla destino:", TABLA_BRONZE)

Archivo fuente: BD eventos riesgo operacional.csv
Lote de procesamiento: bronze_20261120_200301
Tabla destino: eventosriesgoyvar.bronze.eventos_riesgo_raw


In [0]:
# Lectura cruda: todo como string, sin inferir tipos ni limpiar valores.
# Spark resuelve los encabezados duplicados (FUENTE / Fuente) sufijándolos con su posición (FUENTE39, Fuente56).
df_raw = (spark.read
    .option("header", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .option("inferSchema", False)
    .csv(RUTA_CSV))

# Normalización mínima de NOMBRES (exigencia de Delta); los VALORES quedan intactos
mapa_nombres = {c: re.sub(r"[^0-9a-zA-Z]+", "_", c.strip().lower()).strip("_") for c in df_raw.columns}

df_bronze = df_raw.select(
    [F.col("`" + c + "`").alias(mapa_nombres[c]) for c in df_raw.columns]
    + [F.lit(NOMBRE_ARCHIVO).alias("_archivo_origen"),
       F.lit(FECHA_INGESTA).cast("timestamp").alias("_fecha_ingesta"),
       F.lit(LOTE).alias("_lote_procesamiento")])

filas_fuente = df_bronze.count()
print("Filas leídas de la fuente:", filas_fuente, "| Columnas:", len(df_bronze.columns), "(58 de origen + 3 de trazabilidad)")
display(df_bronze.limit(5))

Filas leídas de la fuente: 51889 | Columnas: 61 (58 de origen + 3 de trazabilidad)


fechainicio,horainicio,minutoinicio,fechafinal,horafin,minutofin,fechadescubrimiento,horades,minutodes,tipo,fecha_contabilizacion,horacon,minutocon,fecha_recuperaci_n,horarec,minutorec,moneda,cuantia_bruta,cuantia_total_recuperada,cuantia_recuperada_seguros,cuantia_otras_recuperaciones,cuantia_neta,cod_categoria_n1,categoria_n1,codigo_subcategoria_n2,subcategoria_n2,codigo_subcategoria_n3,subcategoria_riesgo_n3,codigoproducto,producto,codigoperdida,codigosisgeo,ciudad,codigosucursal,sucursal,canal,codigoimpacto,impacto,codigofuente,fuente39,codigo_linea_operativa_n1,linea_operativa_n1,codigo_linea_operativa_n2,linea_operativa_n2,codestror,codproceso,proceso,codactiviadad,actividad,codriesgo,codcausa,cuenta_p_rdida_8_digitos,documento_p_rdida,cuenta_recuperaci_n_8_digitos,documento_recuperaci_n,clase_de_riesgo,fuente56,idinsumo,_archivo_origen,_fecha_ingesta,_lote_procesamiento
05/01/2026,14,45,08/01/2026,15,30,05/01/2026,14,45,P,null,null,null,null,null,null,PESOS,400.000,-,-,-,400.000,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,11,CONFIDIARIO,B,15001000,TUNJA,72,TIBASOSA,CAJERO ELECTRÓNICO,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,1",BANCA MINORISTA,1.1.9.4.1,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.12,Cajeros automáticos,8,624,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6022236,BD eventos riesgo operacional.csv,2026-11-20T20:03:01.809Z,bronze_20261120_200301
13/01/2026,21,29,17/01/2026,11,44,13/01/2026,21,29,P,null,null,null,null,null,null,PESOS,600.000,-,-,-,600.000,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,119,CONFINOMINA,B,05001000,MEDELLIN,20,CASTILLA,CAJERO ELECTRÓNICO,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,1",BANCA MINORISTA,1.1.9.4.1,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.12,Cajeros automáticos,8,624,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6045897,BD eventos riesgo operacional.csv,2026-11-20T20:03:01.809Z,bronze_20261120_200301
02/01/2026,09,25,25/02/2026,08,19,02/01/2026,09,25,P,null,null,null,null,null,null,PESOS,156.800,-,-,-,156.800,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,119,CONFINOMINA,B,11001000,BOGOTA,98,LA SOLEDAD,POS,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,3",SERVICIOS DE TARJETAS DE CRÉDITO Y/O DÉBITO,1.1.9.4.3,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.14,POS,8,627,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6013564,BD eventos riesgo operacional.csv,2026-11-20T20:03:01.809Z,bronze_20261120_200301
05/01/2026,10,25,06/01/2026,17,49,05/01/2026,10,25,P,null,null,null,null,null,null,PESOS,140.000,-,-,-,140.000,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,119,CONFINOMINA,B,05001000,MEDELLIN,39,BARRIO ANTIOQUIA,CAJERO ELECTRÓNICO,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,1",BANCA MINORISTA,1.1.9.4.1,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.12,Cajeros automáticos,8,624,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6019795,BD eventos riesgo operacional.csv,2026-11-20T20:03:01.809Z,bronze_20261120_200301
02/01/2026,10,53,06/01/2026,17,45,02/01/2026,10,53,P,null,null,null,null,null,null,PESOS,400.000,-,-,-,400.000,6,FALLAS TECNOLÓGICAS,6.1,SISTEMAS,6.1.3,TELECOMUNICACIONES,11,CONFIDIARIO,B,05001000,MEDELLIN,18,SAN CRISTOBAL,CAJERO ELECTRÓNICO,5,PERDIDA DE CLIENTES,3,TECNOLOGIA,3,Banca Personal y Minorista,"3,1",BANCA MINORISTA,1.1.9.4.1,6.4,"GESTIÓN DE PRODUCTOS, SERVICIOS Y CANALES",6.4.12,Cajeros automáticos,8,624,null,null,null,null,RIESGO OPERACIONAL,RECLAMACIONES TC Y TD,6014557,BD eventos riesgo operacional.csv,2026-11-20T20:03:01.809Z,bronze_20261120_200301


In [0]:
# Ingesta idempotente: no inserta si este archivo ya fue cargado (evita duplicados por la misma carga)
def tabla_existe(nombre):
    """Verificación eager: en Spark Connect spark.table() es perezoso y no falla
    hasta ejecutar una acción, por eso se fuerza el análisis con .limit(0).count()."""
    try:
        spark.table(nombre).limit(0).count()
        return True
    except Exception:
        return False

ya_cargadas = 0
if tabla_existe(TABLA_BRONZE):
    ya_cargadas = spark.table(TABLA_BRONZE).filter(F.col("_archivo_origen") == NOMBRE_ARCHIVO).count()

if ya_cargadas == 0:
    df_bronze.write.mode("append").saveAsTable(TABLA_BRONZE)
    print(f"Ingesta ejecutada: lote {LOTE} -> {filas_fuente} filas insertadas en {TABLA_BRONZE}")
elif ya_cargadas == filas_fuente:
    print(f"El archivo '{NOMBRE_ARCHIVO}' ya estaba cargado ({ya_cargadas} filas). No se inserta nada: ejecución idempotente.")
else:
    print(f"ADVERTENCIA: '{NOMBRE_ARCHIVO}' ya está cargado con {ya_cargadas} filas pero la fuente tiene {filas_fuente}. No se inserta nada; revisar manualmente (¿cambió el archivo?).")

El archivo 'BD eventos riesgo operacional.csv' ya estaba cargado (51889 filas). No se inserta nada: ejecución idempotente.


In [0]:
# Verificación 1 y 2: conteo y columnas vs fuente
bronze = spark.table(TABLA_BRONZE)
filas_bronze = bronze.count()
print("Filas en Bronze:", filas_bronze, "| Filas en la fuente:", filas_fuente, "| ¿Coinciden?", filas_bronze == filas_fuente)
print("Columnas en Bronze:", len(bronze.columns), "(58 de origen + 3 de trazabilidad)")

# Verificación 3: fidelidad de los valores — anti-join bidireccional sobre las 58 columnas de origen
cols_origen = [mapa_nombres[c] for c in df_raw.columns]
fuente_norm = df_raw.select([F.col("`" + c + "`").alias(mapa_nombres[c]) for c in df_raw.columns])
solo_en_fuente = fuente_norm.exceptAll(bronze.select(cols_origen)).count()
solo_en_bronze = bronze.select(cols_origen).exceptAll(fuente_norm).count()
print("Filas de la fuente que NO están en Bronze:", solo_en_fuente)
print("Filas de Bronze que NO están en la fuente:", solo_en_bronze)

# Verificación 4: ninguna fila se elimina en Bronze. Las filas que comparten valores idénticos
# no se consideran duplicadas (estructura de la BD, ver aclaración) y también se conservan.
filas_unicas_por_valores = bronze.select(cols_origen).dropDuplicates().count()
print("Filas en Bronze:", filas_bronze, "| Filas distintas por valores:", filas_unicas_por_valores,
      "-> se conservan todas (registros independientes)")

# Verificación 5: trazabilidad de la carga
bronze.groupBy("_archivo_origen", "_lote_procesamiento") \
      .agg(F.count("*").alias("filas"), F.min("_fecha_ingesta").alias("primera_ingesta"), F.max("_fecha_ingesta").alias("ultima_ingesta")) \
      .show(truncate=False)

Filas en Bronze: 51889 | Filas en la fuente: 51889 | ¿Coinciden? True
Columnas en Bronze: 61 (58 de origen + 3 de trazabilidad)
Filas de la fuente que NO están en Bronze: 0
Filas de Bronze que NO están en la fuente: 0
Filas en Bronze: 51889 | Filas distintas por valores: 50860 -> se conservan todas (registros independientes)
+---------------------------------+----------------------+-----+--------------------------+--------------------------+
|_archivo_origen                  |_lote_procesamiento   |filas|primera_ingesta           |ultima_ingesta            |
+---------------------------------+----------------------+-----+--------------------------+--------------------------+
|BD eventos riesgo operacional.csv|bronze_20261120_194928|51889|2026-11-20 19:49:28.194432|2026-11-20 19:49:28.194432|
+---------------------------------+----------------------+-----+--------------------------+--------------------------+



In [0]:
# ACLARACIÓN (20-Nov-2026): la agrupación por los valores de las 58 columnas NO es consistente
# con la información y estructura de la BD. Cada fila es un registro independiente aunque sus
# columnas descriptivas coincidan: NO se consideran duplicadas, no se marcan y no se eliminan.
# Consecuencia: la deduplicación por igualdad de valores queda descartada de las reglas de
# calidad de Silver; toda fila de la fuente debe llegar íntegra a las capas siguientes.

# Cuantificación únicamente como trazabilidad de la composición de la fuente:
grupos_valores = bronze.groupBy(cols_origen).count().where(F.col("count") > 1)
filas_comparten_valores = grupos_valores.agg(F.sum("count").alias("filas")).collect()[0][0]
print("Filas en Bronze:", bronze.count())
print("Filas que comparten la totalidad de sus valores con al menos otra fila:", filas_comparten_valores,
      "-> registros independientes, no duplicados")

Filas en Bronze: 51889
Filas que comparten la totalidad de sus valores con al menos otra fila: 1980 -> registros independientes, no duplicados


## Criterios de verificación de la capa Bronze

1. Conteo de filas en Bronze == conteo de la fuente (51.889).
2. 61 columnas: 58 de origen (string, valores intactos) + 3 de trazabilidad.
3. Anti-join bidireccional sobre las 58 columnas de origen == 0 filas de diferencia.
4. No se elimina ningún registro: las filas que comparten valores idénticos no se consideran duplicadas (estructura de la BD) y también se conservan.
5. Trazabilidad: un único archivo/lote por carga.
6. Re-ejecutar la celda de ingesta no inserta duplicados (guarda de idempotencia).